# Project 2 Milestone 2: PCA Baseline Embedding and Diagnostic Visualization\n\nThis notebook builds the first machine-learning baseline for Project 2.\n\nThe goal is to apply standardized PCA embeddings to several Gaia–LAMOST feature spaces and inspect whether the resulting low-dimensional structure is interpretable using Project 1 diagnostic quantities.\n\nThis milestone does not claim any astrophysical discovery. PCA is used here as a transparent baseline before moving to nonlinear methods such as UMAP or density-based clustering.

## Research Question\n\nCan simple linear dimensionality reduction reveal interpretable structure in Gaia–LAMOST chemo-kinematic feature spaces?

In [ ]:
from pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.decomposition import PCA\n\npd.set_option('display.max_columns', 120)\npd.set_option('display.width', 180)

In [ ]:
DATA_PATH = Path('../data/processed/gaia_lamost_larger_velocity_features.csv')\nFIGURE_DIR = Path('../figures')\nOUTPUT_DIR = Path('../data/processed')\n\nFIGURE_DIR.mkdir(parents=True, exist_ok=True)\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n\ndf = pd.read_csv(DATA_PATH)\ndf.shape

## Feature Spaces\n\nThe feature spaces follow Project 2 Milestone 1.

In [ ]:
feature_spaces = {\n    'photometric_astrometric': [\n        'bp_rp',\n        'absolute_g_mag',\n        'distance_pc',\n        'pm_total',\n        'reduced_pm_g',\n    ],\n    'chemical_stellar': [\n        'feh',\n        'teff',\n        'logg',\n    ],\n    'local_kinematic': [\n        'tangential_velocity_kms',\n        'rv',\n    ],\n    'galactocentric_velocity': [\n        'galcen_vx_kms',\n        'galcen_vy_kms',\n        'galcen_vz_kms',\n        'galcen_vtot_kms',\n    ],\n    'combined_chemo_kinematic': [\n        'feh',\n        'bp_rp',\n        'absolute_g_mag',\n        'tangential_velocity_kms',\n        'rv',\n        'galcen_vx_kms',\n        'galcen_vy_kms',\n        'galcen_vz_kms',\n        'galcen_vtot_kms',\n    ],\n}\n\nfeature_spaces

## PCA Helper Function

In [ ]:
def run_pca_for_feature_space(dataframe, feature_name, columns, n_components=3):\n    clean = dataframe.dropna(subset=columns).copy()\n    X = clean[columns].to_numpy()\n\n    scaler = StandardScaler()\n    X_scaled = scaler.fit_transform(X)\n\n    max_components = min(n_components, X_scaled.shape[1])\n    pca = PCA(n_components=max_components, random_state=42)\n    embedding = pca.fit_transform(X_scaled)\n\n    for i in range(max_components):\n        clean[f'{feature_name}_pc{i + 1}'] = embedding[:, i]\n\n    summary = {\n        'feature_space': feature_name,\n        'n_rows': clean.shape[0],\n        'n_features': len(columns),\n        'features': ', '.join(columns),\n    }\n\n    for i, ratio in enumerate(pca.explained_variance_ratio_, start=1):\n        summary[f'pc{i}_explained_variance_ratio'] = ratio\n\n    summary['cumulative_explained_variance_ratio'] = pca.explained_variance_ratio_.sum()\n\n    return clean, pca, summary

## Run PCA Across Feature Spaces

In [ ]:
pca_results = {}\nsummary_rows = []\n\nfor feature_name, columns in feature_spaces.items():\n    clean, pca, summary = run_pca_for_feature_space(df, feature_name, columns, n_components=3)\n    pca_results[feature_name] = {\n        'data': clean,\n        'pca': pca,\n        'columns': columns,\n    }\n    summary_rows.append(summary)\n\npca_summary = pd.DataFrame(summary_rows)\npca_summary

## Save PCA Explained Variance Summary

In [ ]:
PCA_SUMMARY_PATH = OUTPUT_DIR / 'project2_pca_explained_variance_summary.csv'\npca_summary.to_csv(PCA_SUMMARY_PATH, index=False)\nPCA_SUMMARY_PATH

## Combined Chemo-Kinematic PCA Diagnostics\n\nThe combined chemo-kinematic space is the main Project 2 baseline feature space because it combines chemical, photometric, local kinematic, and Galactocentric velocity information.

In [ ]:
combined_name = 'combined_chemo_kinematic'\ncombined = pca_results[combined_name]['data'].copy()\n\npc1 = f'{combined_name}_pc1'\npc2 = f'{combined_name}_pc2'\n\ncombined[[pc1, pc2, 'feh', 'galcen_vtot_kms', 'chemo_kinematic_candidate']].head()

### PCA Colored by Metallicity

In [ ]:
plt.figure(figsize=(8, 6))\nscatter = plt.scatter(\n    combined[pc1],\n    combined[pc2],\n    c=combined['feh'],\n    s=18,\n    alpha=0.8,\n)\nplt.xlabel('Combined chemo-kinematic PC1')\nplt.ylabel('Combined chemo-kinematic PC2')\nplt.title('Project 2 PCA baseline: combined feature space colored by [Fe/H]')\ncbar = plt.colorbar(scatter)\ncbar.set_label('[Fe/H]')\nplt.tight_layout()\nplt.savefig(FIGURE_DIR / 'project2_pca_combined_chemo_kinematic_by_feh.png', dpi=200)\nplt.show()

### PCA Colored by Galactocentric Total Velocity

In [ ]:
plt.figure(figsize=(8, 6))\nscatter = plt.scatter(\n    combined[pc1],\n    combined[pc2],\n    c=combined['galcen_vtot_kms'],\n    s=18,\n    alpha=0.8,\n)\nplt.xlabel('Combined chemo-kinematic PC1')\nplt.ylabel('Combined chemo-kinematic PC2')\nplt.title('Project 2 PCA baseline: combined feature space colored by Galactocentric velocity')\ncbar = plt.colorbar(scatter)\ncbar.set_label('Galactocentric total velocity [km/s]')\nplt.tight_layout()\nplt.savefig(FIGURE_DIR / 'project2_pca_combined_chemo_kinematic_by_vtot.png', dpi=200)\nplt.show()

### PCA with Project 1 Candidate Flags\n\nProject 1 candidate flags are used only as post-hoc interpretation markers.

In [ ]:
candidate_mask = combined['chemo_kinematic_candidate'].astype(bool)\n\nplt.figure(figsize=(8, 6))\nplt.scatter(\n    combined.loc[~candidate_mask, pc1],\n    combined.loc[~candidate_mask, pc2],\n    s=16,\n    alpha=0.5,\n    label='Non-candidate',\n)\nplt.scatter(\n    combined.loc[candidate_mask, pc1],\n    combined.loc[candidate_mask, pc2],\n    s=45,\n    alpha=0.9,\n    marker='x',\n    label='Project 1 chemo-kinematic candidate',\n)\nplt.xlabel('Combined chemo-kinematic PC1')\nplt.ylabel('Combined chemo-kinematic PC2')\nplt.title('Project 2 PCA baseline with Project 1 candidate markers')\nplt.legend()\nplt.tight_layout()\nplt.savefig(FIGURE_DIR / 'project2_pca_combined_chemo_kinematic_candidates.png', dpi=200)\nplt.show()

## Save Combined PCA Embedding

In [ ]:
combined_output_cols = [\n    'source_id',\n    pc1,\n    pc2,\n    'feh',\n    'bp_rp',\n    'absolute_g_mag',\n    'tangential_velocity_kms',\n    'rv',\n    'galcen_vtot_kms',\n    'metallicity_group',\n    'high_vtan_candidate',\n    'metal_poor_candidate',\n    'chemo_kinematic_candidate',\n]\n\nCOMBINED_PCA_PATH = OUTPUT_DIR / 'project2_combined_chemo_kinematic_pca_embedding.csv'\ncombined[combined_output_cols].to_csv(COMBINED_PCA_PATH, index=False)\nCOMBINED_PCA_PATH

## Milestone 2 Notes\n\n- PCA provides a transparent baseline embedding before nonlinear dimensionality reduction.\n- Candidate markers are used only for post-hoc interpretation.\n- Strong separation in PCA space may indicate feature-space structure, but does not by itself confirm distinct astrophysical populations.\n- Later milestones should compare PCA with UMAP and clustering methods.